# Load One Year of FARS Data (Person-Level)

This notebook loads one year of the NHTSA **Fatality Analysis Reporting System (FARS)** data and builds a single table with **one row per person**.

It **only loads** the data and saves it. Nothing is dropped, encoded, or changed.

**Steps**
1. Pick a year and output format
2. Find the year's data folder
3. Read the three main files: `accident`, `vehicle`, `person`
4. Join vehicle and accident details onto each person
5. Check the result
6. Save the table to the `processed/` folder as Parquet or CSV

## Step 1: Setup, pick a year and output format

- `YEAR`: any year that has a folder in `data/` (currently 2016–2024).
- `SAVE_FORMAT`: `"parquet"` or `"csv"`. This sets the file type saved in Step 6.

`DATA_DIR` assumes this notebook is in the `project/` folder, next to `data/`.

In [1]:
from pathlib import Path
import pandas as pd

YEAR = 2016
SAVE_FORMAT = "parquet"   # "parquet" or "csv"
DATA_DIR = Path("data")

## Step 2: Find the year's folder and files

Each year is in a folder named `FARS<YEAR>NationalCSV`.

File name capitalization changes between years. For example, 2016 uses `Person.CSV` and 2024 uses `person.csv`. The `find_file` function ignores case, so the same code works for every year.

In [2]:
year_dir = DATA_DIR / f"FARS{YEAR}NationalCSV"

def find_file(folder, name):
    """Return the file in `folder` whose name matches `name`, ignoring case."""
    for f in folder.iterdir():
        if f.name.lower() == name.lower():
            return f
    raise FileNotFoundError(f"{name} not found in {folder}")

print(find_file(year_dir, "accident.csv"))
print(find_file(year_dir, "vehicle.csv"))
print(find_file(year_dir, "person.csv"))

data/FARS2016NationalCSV/accident.CSV
data/FARS2016NationalCSV/Vehicle.CSV
data/FARS2016NationalCSV/Person.CSV


## Step 3: Read the three files

| File | One row per | Key columns |
|---|---|---|
| `accident` | crash | `ST_CASE` |
| `vehicle` | vehicle in a crash | `ST_CASE`, `VEH_NO` |
| `person` | person in a crash | `ST_CASE`, `VEH_NO`, `PER_NO` |

Read options:
- `encoding="latin-1"`: some text columns (such as city names) have characters that are not valid UTF-8. Latin-1 reads every byte without errors.
- `low_memory=False`: pandas reads each file in one pass, so each column gets one consistent type. This does not change any values.

In [3]:
def read_fars(name):
    return pd.read_csv(find_file(year_dir, name), encoding="latin-1", low_memory=False)

accident = read_fars("accident.csv")
vehicle  = read_fars("vehicle.csv")
person   = read_fars("person.csv")

print(f"accident: {accident.shape[0]:>7,} rows, {accident.shape[1]} columns")
print(f"vehicle:  {vehicle.shape[0]:>7,} rows, {vehicle.shape[1]} columns")
print(f"person:   {person.shape[0]:>7,} rows, {person.shape[1]} columns")

accident:  34,748 rows, 91 columns
vehicle:   52,714 rows, 193 columns
person:    86,474 rows, 129 columns


## Step 4: Join vehicle and accident details onto each person

We start from `person`, so the result has one row per person.

1. **Add vehicle columns** by matching on `ST_CASE` + `VEH_NO`.
2. **Add accident columns** by matching on `ST_CASE`.

Notes on the joins:
- **`how="left"`** keeps every person, even with no match. People who were not in a vehicle (pedestrians, cyclists) have `VEH_NO = 0`. That doesn't match any vehicle row, so their vehicle columns are empty (`NaN`).
- **`validate="many_to_one"`** makes pandas raise an error if a vehicle or crash appears twice. If that happened, people would be duplicated.
- **Suffixes**: some columns appear in more than one file (for example `STATE`, `MONTH`, `HOUR`, `HARM_EV`). We keep **all** of them. The `person` version keeps its original name, and the copies get `_VEH` or `_ACC` added to the end.

In [4]:
df = person.merge(
    vehicle,
    on=["ST_CASE", "VEH_NO"],
    how="left",
    suffixes=("", "_VEH"),
    validate="many_to_one",
)

df = df.merge(
    accident,
    on="ST_CASE",
    how="left",
    suffixes=("", "_ACC"),
    validate="many_to_one",
)

## Step 5: Check the result

- The number of rows should equal the number of rows in `person`.
- The number of columns should equal the column counts of the three files added together, minus the shared key columns (`ST_CASE` and `VEH_NO` from vehicle, and `ST_CASE` from accident).
- `(ST_CASE, VEH_NO, PER_NO)` should be unique, so each row is exactly one person.

In [5]:
assert len(df) == len(person), "Row count changed during merge"
assert df.shape[1] == person.shape[1] + (vehicle.shape[1] - 2) + (accident.shape[1] - 1), "Unexpected column count"
assert not df.duplicated(["ST_CASE", "VEH_NO", "PER_NO"]).any(), "Duplicate person keys"

n_no_vehicle = (df["VEH_NO"] == 0).sum()
print(f"{YEAR}: {df.shape[0]:,} people x {df.shape[1]} columns")
print(f"People not in a vehicle (VEH_NO = 0): {n_no_vehicle:,}")
df.head()

2016: 86,474 people x 410 columns
People not in a vehicle (VEH_NO = 0): 7,661


,STATE,STATENAME,ST_CASE,VE_FORMS,VEH_NO,PER_NO,STR_VEH,COUNTY,DAY,DAYNAME,...,HOSP_MN,HOSP_MNNAME,CF1,CF1NAME,CF2,CF2NAME,CF3,CF3NAME,FATALS,DRUNK_DR
0,1,Alabama,10001,1,1,1,0,73,1,1,...,88,Not Applicable (Not Transported),0,NaN,0,NaN,0,NaN,1,1
1,1,Alabama,10002,1,1,1,0,73,10,10,...,99,Unknown EMS Hospital Arrival Time,0,NaN,0,NaN,0,NaN,1,1
2,1,Alabama,10002,1,1,2,0,73,10,10,...,99,Unknown EMS Hospital Arrival Time,0,NaN,0,NaN,0,NaN,1,1
3,1,Alabama,10003,1,1,1,0,73,1,1,...,88,Not Applicable (Not Transported),0,NaN,0,NaN,0,NaN,1,0
4,1,Alabama,10003,1,2,1,0,73,1,1,...,88,Not Applicable (Not Transported),0,NaN,0,NaN,0,NaN,1,0


## Step 6: Save the table

Write the table to `processed/fars_<YEAR>_person.parquet` or `processed/fars_<YEAR>_person.csv`, depending on `SAVE_FORMAT`. The `processed/` folder sits next to `data/` and is created if it doesn't exist.

**Parquet** (recommended):
- **Keeps column types.** The data comes back exactly as it was saved.
- **Smaller and faster.** A 2016 file is about 18 MB, compared with about 300 MB as a CSV.
- **Load only what you need.** `pd.read_parquet(path, columns=[...])` reads just those columns, which saves RAM.
- Needs the `pyarrow` package (`pip install pyarrow`).

**CSV**:
- Opens in anything, including Excel and a text editor.
- Stores everything as text, so pandas has to guess the column types again when it reads the file.

Both use `index=False`, which leaves out pandas' row numbers so the file has exactly the same columns as `df`. Running the notebook again for the same year and format overwrites that file.

In [6]:
OUT_DIR = Path("processed")
OUT_DIR.mkdir(exist_ok=True)

out_path = OUT_DIR / f"fars_{YEAR}_person.{SAVE_FORMAT}"

if SAVE_FORMAT == "parquet":
    df.to_parquet(out_path, index=False)
elif SAVE_FORMAT == "csv":
    df.to_csv(out_path, index=False)
else:
    raise ValueError('SAVE_FORMAT must be "parquet" or "csv"')

print(f"Saved {out_path} ({out_path.stat().st_size / 1e6:.0f} MB)")

Saved processed/fars_2016_person.parquet (18 MB)
